# FPO Training on Mujoco Environments

This notebook clones the FPO repository, installs dependencies, trains FPO on Ant, Humanoid, Hopper, Walker, and HalfCheetah environments, and displays training metrics and graphs for rewards and episode lengths.

## 1. Clone the Repository

In [ ]:
!git clone https://github.com/manavparikh2203/fpo.git /content/fpo
%cd /content/fpo

## 2. Install Dependencies

In [ ]:
!pip install -e playground
!pip install "gymnasium[mujoco]" optax tqdm

## 3. Set Up the Environment

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
import gymnasium as gym

env = gym.make('Ant-v5')
obs, info = env.reset()

print('Gymnasium Ant-v5 created successfully')
env.close()

print('Observation shape:', obs.shape)
print('Action space:', env.action_space)

## 4. Run the Training Script

In [ ]:
# Live FPO Training on Gymnasium Ant-v5 with Real-Time Plotting
import os
os.environ['MUJOCO_GL'] = 'egl'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

import argparse
from pathlib import Path

import gymnasium as gym
import jax
import jax.numpy as jnp
import jax.tree_util as tree_util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

from flow_policy import fpo, rollouts

# Copy the training code here for live plotting
# (Same as train_fpo_gymnasium_ant.py but with live updates)

def make_gym_ant_env(env_name: str, num_envs: int):
    base_env = gym.vector.SyncVectorEnv([lambda: gym.make(env_name) for _ in range(num_envs)])
    class GymAntEnv:
        def __init__(self, env):
            self.env = env
            self.action_space = env.single_action_space
            self.observation_space = env.single_observation_space
            self.observation_size = int(np.prod(self.observation_space.shape))
            self.action_size = int(np.prod(self.action_space.shape))
        def reset(self):
            obs, _ = self.env.reset()
            return np.asarray(obs, dtype=np.float32)
        def step(self, action: np.ndarray):
            action = np.asarray(action, dtype=np.float32)
            action = np.clip(action, self.action_space.low, self.action_space.high)
            obs, reward, terminated, truncated, infos = self.env.step(action)
            return (
                np.asarray(obs, dtype=np.float32),
                np.asarray(reward, dtype=np.float32),
                np.asarray(terminated, dtype=np.bool_),
                np.asarray(truncated, dtype=np.bool_),
                infos,
            )
    return GymAntEnv(base_env)

def extract_episode_stats(rewards, truncation, discount):
    episode_rewards = []
    episode_lengths = []
    T, B = rewards.shape
    for b in range(B):
        current_reward = 0.0
        current_length = 0
        for t in range(T):
            current_reward += float(rewards[t, b])
            current_length += 1
            if truncation[t, b] > 0.5 or discount[t, b] == 0.0:
                episode_rewards.append(current_reward)
                episode_lengths.append(current_length)
                current_reward = 0.0
                current_length = 0
    return np.array(episode_rewards, dtype=np.float32), np.array(episode_lengths, dtype=np.int32)

def collect_transitions(env, agent_state, prng, config):
    obs = jnp.asarray(env.reset(), dtype=jnp.float32)
    obs_seq = []
    next_obs_seq = []
    action_seq = []
    reward_seq = []
    truncation_seq = []
    discount_seq = []
    action_info_seq = []
    for t in range(config.iterations_per_env):
        prng, prng_step = jax.random.split(prng)
        action, action_info = agent_state.sample_action(obs, prng_step, deterministic=False)
        next_obs_np, reward_np, terminated_np, truncated_np, _ = env.step(np.asarray(action))
        next_obs = jnp.asarray(next_obs_np, dtype=jnp.float32)
        reward = jnp.asarray(reward_np, dtype=jnp.float32)
        truncation = jnp.asarray(truncated_np, dtype=jnp.float32)
        discount = 1.0 - jnp.asarray(terminated_np, dtype=jnp.float32)
        obs_seq.append(obs)
        next_obs_seq.append(next_obs)
        action_seq.append(action)
        reward_seq.append(reward)
        truncation_seq.append(truncation)
        discount_seq.append(discount)
        action_info_seq.append(action_info)
        obs = next_obs
    transitions = rollouts.TransitionStruct(
        obs=jnp.stack(obs_seq),
        next_obs=jnp.stack(next_obs_seq),
        action=jnp.stack(action_seq),
        action_info=tree_util.tree_map(lambda *xs: jnp.stack(xs), *action_info_seq),
        reward=jnp.stack(reward_seq),
        truncation=jnp.stack(truncation_seq),
        discount=jnp.stack(discount_seq),
    )
    return transitions, prng

# Training parameters
env_name = 'Ant-v5'
output_dir = Path('./gym_fpo_results') / 'Antv5'
output_dir.mkdir(parents=True, exist_ok=True)
num_timesteps = 1000000
num_envs = 16
batch_size = 512
num_minibatches = 8
unroll_length = 30
num_updates_per_batch = 4
seed = 42

config = fpo.FpoConfig(
    num_timesteps=num_timesteps,
    num_envs=num_envs,
    batch_size=batch_size,
    num_minibatches=num_minibatches,
    unroll_length=unroll_length,
    num_updates_per_batch=num_updates_per_batch,
)

env = make_gym_ant_env(env_name, num_envs)
agent_state = fpo.FpoState.init(prng=jax.random.key(seed), env=env, config=config)
prng = jax.random.key(seed + 1)

outer_iters = int(num_timesteps // (config.iterations_per_env * num_envs))
history = []

for i in range(outer_iters):
    transitions, prng = collect_transitions(env, agent_state, prng, config)
    agent_state, metrics = agent_state.training_step(transitions)

    reward_np = np.asarray(transitions.reward)
    truncation_np = np.asarray(transitions.truncation)
    discount_np = np.asarray(transitions.discount)
    episode_rewards, episode_lengths = extract_episode_stats(reward_np, truncation_np, discount_np)

    mean_reward = float(np.mean(reward_np))
    mean_episode_reward = float(np.mean(episode_rewards)) if episode_rewards.size else mean_reward
    mean_episode_length = float(np.mean(episode_lengths)) if episode_lengths.size else 0.0

    row = {
        "step": i,
        "mean_reward": mean_reward,
        "mean_episode_reward": mean_episode_reward,
        "mean_episode_length": mean_episode_length,
        "num_episodes": int(episode_rewards.size),
        **{f"train_{k}": float(np.mean(v)) for k, v in metrics.items()},
    }
    history.append(row)

    print(f"Iter {i+1}/{outer_iters} | mean_reward={mean_reward:.3f} | mean_episode_reward={mean_episode_reward:.3f} | episodes={row['num_episodes']}")

    # Live plot every 10 iterations
    if (i + 1) % 10 == 0 or i == outer_iters - 1:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot([row["step"] for row in history], [row["mean_reward"] for row in history], label="mean_reward")
        ax.plot([row["step"] for row in history], [row["mean_episode_reward"] for row in history], label="mean_episode_reward")
        ax.set_title(f"{env_name} FPO Training (Iter {i+1})")
        ax.set_xlabel("outer iteration")
        ax.set_ylabel("reward")
        ax.legend()
        display(fig)
        plt.close(fig)

# Final save
pd.DataFrame(history).to_csv(output_dir / "train_metrics.csv", index=False)
print(f"Training complete. Results saved to {output_dir}")

In [ ]:
# Live FPO Training on Gymnasium Ant-v5 with Real-Time Plotting
import os
os.environ['MUJOCO_GL'] = 'egl'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

import argparse
from pathlib import Path

import gymnasium as gym
import jax
import jax.numpy as jnp
import jax.tree_util as tree_util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

from flow_policy import fpo, rollouts

# Copy the training code here for live plotting
# (Same as train_fpo_gymnasium_ant.py but with live updates)

def make_gym_ant_env(env_name: str, num_envs: int):
    base_env = gym.vector.SyncVectorEnv([lambda: gym.make(env_name) for _ in range(num_envs)])
    class GymAntEnv:
        def __init__(self, env):
            self.env = env
            self.action_space = env.single_action_space
            self.observation_space = env.single_observation_space
            self.observation_size = int(np.prod(self.observation_space.shape))
            self.action_size = int(np.prod(self.action_space.shape))
        def reset(self):
            obs, _ = self.env.reset()
            return np.asarray(obs, dtype=np.float32)
        def step(self, action: np.ndarray):
            action = np.asarray(action, dtype=np.float32)
            action = np.clip(action, self.action_space.low, self.action_space.high)
            obs, reward, terminated, truncated, infos = self.env.step(action)
            return (
                np.asarray(obs, dtype=np.float32),
                np.asarray(reward, dtype=np.float32),
                np.asarray(terminated, dtype=np.bool_),
                np.asarray(truncated, dtype=np.bool_),
                infos,
            )
    return GymAntEnv(base_env)

def extract_episode_stats(rewards, truncation, discount):
    episode_rewards = []
    episode_lengths = []
    T, B = rewards.shape
    for b in range(B):
        current_reward = 0.0
        current_length = 0
        for t in range(T):
            current_reward += float(rewards[t, b])
            current_length += 1
            if truncation[t, b] > 0.5 or discount[t, b] == 0.0:
                episode_rewards.append(current_reward)
                episode_lengths.append(current_length)
                current_reward = 0.0
                current_length = 0
    return np.array(episode_rewards, dtype=np.float32), np.array(episode_lengths, dtype=np.int32)

def collect_transitions(env, agent_state, prng, config):
    obs = jnp.asarray(env.reset(), dtype=jnp.float32)
    obs_seq = []
    next_obs_seq = []
    action_seq = []
    reward_seq = []
    truncation_seq = []
    discount_seq = []
    action_info_seq = []
    for t in range(config.iterations_per_env):
        prng, prng_step = jax.random.split(prng)
        action, action_info = agent_state.sample_action(obs, prng_step, deterministic=False)
        next_obs_np, reward_np, terminated_np, truncated_np, _ = env.step(np.asarray(action))
        next_obs = jnp.asarray(next_obs_np, dtype=jnp.float32)
        reward = jnp.asarray(reward_np, dtype=jnp.float32)
        truncation = jnp.asarray(truncated_np, dtype=jnp.float32)
        discount = 1.0 - jnp.asarray(terminated_np, dtype=jnp.float32)
        obs_seq.append(obs)
        next_obs_seq.append(next_obs)
        action_seq.append(action)
        reward_seq.append(reward)
        truncation_seq.append(truncation)
        discount_seq.append(discount)
        action_info_seq.append(action_info)
        obs = next_obs
    transitions = rollouts.TransitionStruct(
        obs=jnp.stack(obs_seq),
        next_obs=jnp.stack(next_obs_seq),
        action=jnp.stack(action_seq),
        action_info=tree_util.tree_map(lambda *xs: jnp.stack(xs), *action_info_seq),
        reward=jnp.stack(reward_seq),
        truncation=jnp.stack(truncation_seq),
        discount=jnp.stack(discount_seq),
    )
    return transitions, prng

# Training parameters
env_name = 'Ant-v5'
output_dir = Path('./gym_fpo_results') / 'Antv5'
output_dir.mkdir(parents=True, exist_ok=True)
num_timesteps = 1000000
num_envs = 16
batch_size = 512
num_minibatches = 8
unroll_length = 30
num_updates_per_batch = 4
seed = 42

config = fpo.FpoConfig(
    num_timesteps=num_timesteps,
    num_envs=num_envs,
    batch_size=batch_size,
    num_minibatches=num_minibatches,
    unroll_length=unroll_length,
    num_updates_per_batch=num_updates_per_batch,
)

env = make_gym_ant_env(env_name, num_envs)
agent_state = fpo.FpoState.init(prng=jax.random.key(seed), env=env, config=config)
prng = jax.random.key(seed + 1)

outer_iters = int(num_timesteps // (config.iterations_per_env * num_envs))
history = []

for i in range(outer_iters):
    transitions, prng = collect_transitions(env, agent_state, prng, config)
    agent_state, metrics = agent_state.training_step(transitions)

    reward_np = np.asarray(transitions.reward)
    truncation_np = np.asarray(transitions.truncation)
    discount_np = np.asarray(transitions.discount)
    episode_rewards, episode_lengths = extract_episode_stats(reward_np, truncation_np, discount_np)

    mean_reward = float(np.mean(reward_np))
    mean_episode_reward = float(np.mean(episode_rewards)) if episode_rewards.size else mean_reward
    mean_episode_length = float(np.mean(episode_lengths)) if episode_lengths.size else 0.0

    row = {
        "step": i,
        "mean_reward": mean_reward,
        "mean_episode_reward": mean_episode_reward,
        "mean_episode_length": mean_episode_length,
        "num_episodes": int(episode_rewards.size),
        **{f"train_{k}": float(np.mean(v)) for k, v in metrics.items()},
    }
    history.append(row)

    print(f"Iter {i+1}/{outer_iters} | mean_reward={mean_reward:.3f} | mean_episode_reward={mean_episode_reward:.3f} | episodes={row['num_episodes']}")

    # Live plot every 10 iterations
    if (i + 1) % 10 == 0 or i == outer_iters - 1:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot([row["step"] for row in history], [row["mean_reward"] for row in history], label="mean_reward")
        ax.plot([row["step"] for row in history], [row["mean_episode_reward"] for row in history], label="mean_episode_reward")
        ax.set_title(f"{env_name} FPO Training (Iter {i+1})")
        ax.set_xlabel("outer iteration")
        ax.set_ylabel("reward")
        ax.legend()
        display(fig)
        plt.close(fig)

# Final save
pd.DataFrame(history).to_csv(output_dir / "train_metrics.csv", index=False)
print(f"Training complete. Results saved to {output_dir}")

**Note:** The Gymnasium Ant-v5 FPO training is handled by the previous cell running `train_fpo_gymnasium_ant.py`. The old playground batch script is not used for this Gymnasium-only workflow.

## 5. Display Training Metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

output_dir = Path('./gym_fpo_results') / 'Antv5'

train_csv = output_dir / 'train_metrics.csv'
if train_csv.exists():
    df = pd.read_csv(train_csv)
    print('Gymnasium Ant-v5 FPO training metrics:')
    print(df.head())
    print('\n')
else:
    print('No training metrics found at', train_csv)

## 6. Plot Graphs

In [ ]:
from IPython.display import Image, display

output_dir = Path('./gym_fpo_results') / 'Antv5'

reward_png = output_dir / 'reward_curve.png'
if reward_png.exists():
    display(Image(filename=str(reward_png)))
else:
    print('No reward plot found at', reward_png)